# 07 -- Farm hands: hire and tend a second loop

Both leaderboard replays analyzed in `PROGRESS.md` (2026-08-13 HechtJP, 2026-08-20 Muhammed Ali Kul) show the opponent using multiple farm hands as the single biggest lever we're missing -- the latter held up to 10 hands vs. our 0. This notebook adds the first increment: hire hands daily and give them a tile loop to tend, same priority logic as `wheat_loop_12_agent` (harvest > water > plant > move).

Facts confirmed before/while writing code (see source or the mini-tests below, not assumed):
- `kaggriculture.py::_do_hire` / `_hire_cost`: no hard cap on hand count in the engine -- only gated by money (Fibonacci cost, resets daily) and `maxMarketOrdersPerTurn` (10/turn, shared with other market orders). Hiring is effectively free: 5 hands in one day costs `1+1+2+3+5 = 12`.
- `kaggriculture.py::_drop_inventories_to_shed`, called from `_end_of_day`: **every** unit's field inventory (farmer's and each hand's) is auto-dropped into the shared shed at end of day. Hands never need to visit the shed themselves -- a loop far from `(4,4)` is fine.
- Hands do **not** all spawn at `(4,4)` -- see the "Finding" section below, discovered via the mini-tests. They land on whichever of the 4 shed corners is least occupied, which usually isn't `(4,4)` since the farmer sits there permanently.

In [1]:
import sys
from pathlib import Path

from kaggle_environments import make

sys.path.insert(0, str(Path.cwd().parent / "src"))

from kaggriculture_bot.evaluate import evaluate
from kaggriculture_bot.strategies.baseline import wheat_loop_12_agent, LOOP_ORDER_12, DIRECTIONS

## A second loop for the hands to share

`LOOP_ORDER_12` (the farmer's loop) already claims the 12 tiles
closest to the shed. This is a second, 16-tile loop built the same way
(rectangle + detour back to the hub) covering 11 of the 13
previously-untended NW tiles (`(0,2)`/`(1,2)` still uncovered -- known
gap, not today's problem). It shares 5 tiles with `LOOP_ORDER_12` as
pure transit on the way out/back, which is harmless -- units can
co-occupy a tile.

All 5 hands will share this *same* dict (see markdown cell above the
agent function for the redundancy trade-off this implies).

In [2]:
LOOP_ORDER_HANDS = {
    (4, 4): (3, 4),
    (3, 4): (3, 3),
    (3, 3): (3, 2),
    (3, 2): (3, 1),
    (3, 1): (2, 1),
    (2, 1): (1, 1),
    (1, 1): (0, 1),
    (0, 1): (0, 0),
    (0, 0): (1, 0),
    (1, 0): (2, 0),
    (2, 0): (3, 0),
    (3, 0): (4, 0),
    (4, 0): (4, 1),
    (4, 1): (4, 2),
    (4, 2): (4, 3),
    (4, 3): (4, 4),
}

## Sanity check: valid closed cycle?

Same check as `notebooks/06_farmer_movement_12_tile_loop.ipynb` /
`tests/test_baseline.py::assert_is_closed_adjacent_cycle`.

In [3]:
def assert_is_closed_adjacent_cycle(loop_order):
    n = len(loop_order)
    start = next(iter(loop_order))
    pos = start
    visited = set()
    for _ in range(n):
        visited.add(pos)
        (x, y), (nx, ny) = pos, loop_order[pos]
        assert abs(nx - x) + abs(ny - y) == 1
        pos = (nx, ny)
    assert pos == start
    assert visited == set(loop_order)


assert_is_closed_adjacent_cycle(LOOP_ORDER_HANDS)
print(f"LOOP_ORDER_HANDS: {len(LOOP_ORDER_HANDS)} tiles, valid closed cycle")
print("overlap with farmer's loop:", set(LOOP_ORDER_HANDS) & set(LOOP_ORDER_12))

LOOP_ORDER_HANDS: 16 tiles, valid closed cycle
overlap with farmer's loop: {(4, 4), (3, 4), (4, 3), (3, 3), (3, 2)}


## Mini-tests: poke at a real `obs` before writing the function

A throwaway agent that does nothing but hire 5 hands on the first turn
of every day and `PASS` otherwise (one `PASS` per hand currently
alive, so the action dict always matches `len(hands)` -- an
`action["hands"]` shorter than the real hand count is untested
territory here, don't assume it's a safe no-op). Playing this against
`"random"` for a few turns gives real `obs` dicts to inspect below,
instead of guessing at the shape.

In [4]:
def _test_hire_agent(obs):
    me = obs["farms"][obs["player"]]
    market = [["HIRE"]] * 5 if obs["hour"] == 0 else []
    hands = [["PASS"]] * len(me["hands"])
    return {"farmer": ["PASS"], "hands": hands, "market": market}


test_env = make("kaggriculture", configuration={"episodeSteps": 720}, debug=True)
test_env.run([_test_hire_agent, "random"])
print("steps recorded:", len(test_env.steps))

steps recorded: 720


In [5]:
# Pick any turn to inspect -- e.g. turn 0 (before any HIRE resolves),
# turn 1 (right after the HIRE order from turn 0 should resolve), turn 2, ...
obs = test_env.steps[1][0]["observation"]
print("hour:", obs["hour"], "hires_today:", obs["farms"][obs["player"]]["hires_today"])
print("hands:", obs["farms"][obs["player"]]["hands"])
len(obs["farms"][obs["player"]]["hands"])

hour: 1 hires_today: 5
hands: [[5, 4], [4, 5], [5, 5], [4, 4], [5, 4]]


5

In [6]:
# Confirm the hire lag and hand ordering across the first few turns of day 0.
for i in range(6):
    o = test_env.steps[i][0]["observation"]
    print(i, "hour:", o["hour"], "hands:", o["farms"][o["player"]]["hands"])

0 hour: 0 hands: []
1 hour: 1 hands: [[5, 4], [4, 5], [5, 5], [4, 4], [5, 4]]
2 hour: 2 hands: [[5, 4], [4, 5], [5, 5], [4, 4], [5, 4]]
3 hour: 3 hands: [[5, 4], [4, 5], [5, 5], [4, 4], [5, 4]]
4 hour: 4 hands: [[5, 4], [4, 5], [5, 5], [4, 4], [5, 4]]
5 hour: 5 hands: [[5, 4], [4, 5], [5, 5], [4, 4], [5, 4]]


### Finding: hands do NOT spawn at `(4,4)`

`kaggriculture.py::_spawn_hand` places each new hand at the least-occupied of the 4 shed-access corners (`(4,4)` NW, `(5,4)` NE, `(4,5)` SW, `(5,5)` SE), tie-broken NWSE. Since the farmer permanently occupies `(4,4)`, that corner starts with a 1-occupant head start -- so new hands are routed to the *other three* corners first. Confirmed above: of 5 hands hired, only 1 landed on `(4,4)`; the other 4 landed on `(5,4)`/`(4,5)`/`(5,5)` -- NE/SW/SE corners, `"LOCKED"` unless that quadrant is bought (confirmed via `tiles[y][x]` below).

This breaks two things in the plan as it stood:

1. **A real bug**: `wheat_loop_12_agent`'s tend logic (`tile.get("kind") == "PLANT"`) assumes `tile` is `None` or a dict. `tile == "LOCKED"` is a *string* -- `.get()` on it raises `AttributeError`. Without an explicit guard, 4 of 5 hands crash on their first turn. `wheat_loop_hands_agent` needs to check for `"LOCKED"` before checking `.get("kind")`, and just treat it like "nothing to tend here, move on" (same branch as the empty/no-seed case).
2. **`LOOP_ORDER_HANDS` is missing keys**: it only has `(4,4)` as an entry point. A hand spawning at `(5,4)`/`(4,5)`/`(5,5)` needs a valid next-step from wherever it lands, or it KeyErrors on its first move.

The fix is cheap, though: all 4 shed corners are mutually adjacent (a 2x2 block), so add the 3 missing corners to `LOOP_ORDER_HANDS`, each routed one step into `(4,4)` (already the loop's entry point) -- e.g. `(5,4): (4,4), (4,5): (4,4), (5,5): (4,4)`. Costs at most 1 extra turn for a hand that didn't spawn directly on `(4,4)`, not a real commute.

In [7]:
tiles = obs["farms"][obs["player"]]["tiles"]
for corner in [(4, 4), (5, 4), (4, 5), (5, 5)]:
    x, y = corner
    print(corner, "->", repr(tiles[y][x]))

(4, 4) -> None
(5, 4) -> 'LOCKED'
(4, 5) -> 'LOCKED'
(5, 5) -> 'LOCKED'


In [8]:
SPAWN_FEEDER = {(5, 4): (4, 4), (4, 5): (4, 4), (5, 5): (5, 4)}
for pos, nxt in SPAWN_FEEDER.items():
    assert abs(nxt[0] - pos[0]) + abs(nxt[1] - pos[1]) == 1
print("SPAWN_FEEDER: valid, all one step closer to LOOP_ORDER_HANDS's entry point (4,4)")

SPAWN_FEEDER: valid, all one step closer to LOOP_ORDER_HANDS's entry point (4,4)


## Building `wheat_loop_hands_agent`

Your turn. Three pieces now, reusing what `wheat_loop_12_agent` already does plus the two fixes found above:

1. **Hiring**: while the hand count is below 5 (check `len(obs["farms"][obs["player"]]["hands"])`), queue `["HIRE"]` market orders -- all 5 in the first turn of the day, per the replay pattern we confirmed (batched, not staggered).
2. **Tending**: for the farmer, same priority logic as `wheat_loop_12_agent` (harvest > water > plant > move via `LOOP_ORDER_12`). For *each* hand in `obs["farms"][obs["player"]]["hands"]`, the identical priority logic but looked up against `{**LOOP_ORDER_HANDS, **SPAWN_FEEDER}` instead -- build one op per hand, in the same order as that list (`AGENTS.md`: "one op per hired hand, in hands order"). **Guard for `tile == "LOCKED"` before calling `.get("kind")`** -- treat it the same as the empty-tile "nothing to do, move on" branch, since a hand can land on a locked corner its first turn.
3. Careful with the shared `WHEAT` seed pool (`private["seeds"]`): with up to 6 units (farmer + 5 hands) all wanting to plant, buying 1 seed at a time like `wheat_loop_12_agent` does won't be enough -- most of them will find `wheat_seeds <= 0` and fall through to `PASS` instead of `PLANT`. Worth deciding how many to buy and when.

In [9]:
len(obs["farms"][obs["player"]]["hands"])

5

In [10]:
def wheat_loop_hands_agent(obs):
    wheat_seeds = obs["private"]["seeds"].get("WHEAT", 0)
    
    market = []
    loop_hands = {**LOOP_ORDER_HANDS, **SPAWN_FEEDER}
    hands = []

    for hx, hy in obs["farms"][obs["player"]]["hands"]:
        target_x, target_y = loop_hands[(hx, hy)]
        hand_direction = DIRECTIONS[(target_x - hx, target_y - hy)]
        hand_tile = obs["farms"][obs["player"]]["tiles"][hy][hx]
        if hand_tile == "LOCKED":
            op = [hand_direction]
        elif hand_tile is None and wheat_seeds > 0:
            op = ["PLANT", "WHEAT"]
        elif hand_tile is not None and hand_tile.get("kind") == "PLANT":
            crop_age = obs["day"] - hand_tile["planted_day"]
            if crop_age >= 2:
                op = ["HARVEST"]
            elif not hand_tile.get("watered_today"):
                op = ["WATER"]
            else:
                op = [hand_direction]
        else:
            op = [hand_direction]
        hands.append(op)

    if len(obs["farms"][obs["player"]]["hands"]) < 5:
        market.append(["HIRE"])
    
    if obs["step"] == 0 or wheat_seeds <= 5:
        market.append(["BUY_SEED", "WHEAT", 6 - wheat_seeds])
    
    farmer_pos = obs["farms"][obs["player"]]["farmer"]
    x, y = farmer_pos
    target_x, target_y = LOOP_ORDER_12[(x, y)]
    direction = DIRECTIONS[(target_x - x, target_y - y)]
    
    tile = obs["farms"][obs["player"]]["tiles"][y][x]
    
    wheat_in_shed = obs["private"]["shed"].get("WHEAT", 0)
    if wheat_in_shed > 0:
        market.append(["SELL", "WHEAT", 1])
    
    if tile is None and wheat_seeds > 0:
        farmer = ["PLANT", "WHEAT"]
    elif tile is not None and tile.get("kind") == "PLANT":
        crop_age = obs["day"] - tile["planted_day"]
        if crop_age >= 2:
            farmer = ["HARVEST"]
        elif not tile.get("watered_today"):
            farmer = ["WATER"]
        else:
            farmer = [direction]
    else:
        farmer = [direction]
    
    return {"farmer": farmer, "hands": hands, "market": market}


## Validate against the current best

Per this project's rule (recompute the baseline in the same run): the
agent to beat is `wheat_loop_12_agent`, not further back. Run twice
(separate `evaluate()` calls / fresh `env`s), per the 2026-08-19 harness
note in `PROGRESS.md`.

In [11]:
#print(evaluate(wheat_loop_12_agent, wheat_loop_hands_agent, n=20))

KeyboardInterrupt: 

In [ ]:
#print(evaluate(wheat_loop_12_agent, wheat_loop_hands_agent, n=20))

{'win_rate': 0.0, 'avg_money': 5382.0}


## Bug found after graduation: hands clump into one permanent herd

Playing a full episode and logging `obs["farms"][player]["hands"]` turn by
turn shows something worse than occasional overlap: from a few turns into
day 0 onward, **all 5 hands land on the exact same tile and stay there for
the rest of the day**, moving as a single block. Of the 16 tiles in
`LOOP_ORDER_HANDS`, only whichever one the herd currently occupies gets
tended -- "5 hands" behaves like 1.

**Root cause**: every hand's action is a pure function of its *current
tile* (`loop_hands[(hx, hy)]`, `hand_tile.get(...)`) -- nothing
distinguishes one hand from another. The moment two hands land on the same
tile (easy: they all funnel through the single shared entry point `(4,4)`,
and a hand that's watering/waiting lets a faster one behind it catch up),
they compute the *identical* next action forever after -- a one-turn
coincidence becomes permanent, since the loop is a single path with no
branching and no per-hand state.

For comparison, the real leaderboard replay analyzed in `PROGRESS.md`
(`replays/94568249.json`, Muhammed Ali Kul, farm index 0) also shows
transient overlaps (473/720 turns with >=2 units sharing a tile) -- but
never more than **2 consecutive turns** with every hand on the same tile.
Their hands must have independent per-hand targets, not one shared path.

In [12]:
def hand_position_log(agent, opponent="random", n_steps=720):
    """Run one full episode of `agent` vs `opponent` and return the list of
    hand positions per turn, from `agent`'s own observation."""
    log_env = make("kaggriculture", configuration={"episodeSteps": n_steps}, debug=True)
    log_env.run([agent, opponent])
    return [
        [tuple(h) for h in step[0]["observation"]["farms"][step[0]["observation"]["player"]]["hands"]]
        for step in log_env.steps
    ]


def max_full_collapse_streak(hands_per_turn):
    """Longest run of consecutive turns where every hand sits on the exact
    same tile -- a permanent merge, not a passing coincidence."""
    max_streak = streak = 0
    for hands in hands_per_turn:
        if hands and len(set(hands)) == 1:
            streak += 1
            max_streak = max(max_streak, streak)
        else:
            streak = 0
    return max_streak


hands_log = hand_position_log(wheat_loop_hands_agent)
print("max streak of ALL hands glued to the same tile:", max_full_collapse_streak(hands_log))
print("example, hours 9-13 of day 0:", hands_log[9:14])

max streak of ALL hands glued to the same tile: 17
example, hours 9-13 of day 0: [[(3, 2), (3, 2), (3, 2), (3, 2), (3, 2)], [(3, 2), (3, 2), (3, 2), (3, 2), (3, 2)], [(3, 1), (3, 1), (3, 1), (3, 1), (3, 1)], [(3, 1), (3, 1), (3, 1), (3, 1), (3, 1)], [(3, 1), (3, 1), (3, 1), (3, 1), (3, 1)]]


## Fix: split the loop into 5 disjoint zones, one per hand

Hands don't get a real ID from the engine -- `obs["farms"][player]["hands"]`
is just a list of `[x, y]` positions. But that list's *order* is stable
turn to turn (confirmed above: an existing hand keeps its index; new hands
are appended at the end), so a hand's position in the list can serve as a
de facto ID via `enumerate()`.

Rather than all 5 hands sharing `LOOP_ORDER_HANDS`, each hand index now
gets its own small closed loop, carved out of the same 13 free NW tiles
(the ones `LOOP_ORDER_12`'s farmer doesn't already use -- all of `y=0`/`y=1`
plus `(0,2)`/`(1,2)`/`(4,2)` from `y=2`). The 5 zones are pairwise
disjoint, so once a hand settles into its own zone, it is *structurally*
impossible for it to ever land on the same tile as a different-index hand
again -- not just unlikely, like the opponent's replay, but impossible.
Bonus: this also finally covers `(0,2)`/`(1,2)`, the gap noted as
untended back when `LOOP_ORDER_HANDS` was built.

Each hand still reaches its zone via the same shared `LOOP_ORDER_HANDS` +
`SPAWN_FEEDER` path from spawn -- the zone dict only *overrides* the one
or two tiles where a hand should peel off the shared path into its own
private loop, instead of continuing on it. Hand 4's zone is a single tile
(`(4,2)`) with nowhere to move to, so it needs a `PASS` fallback instead of
a direction lookup once there's nothing left to tend.

In [13]:
HAND_ZONES = [
    {(0, 0): (1, 0), (1, 0): (1, 1), (1, 1): (0, 1), (0, 1): (0, 0)},  # hand 0: 2x2 block
    {(2, 0): (3, 0), (3, 0): (3, 1), (3, 1): (2, 1), (2, 1): (2, 0)},  # hand 1: 2x2 block
    {(4, 0): (4, 1), (4, 1): (4, 0)},  # hand 2: 2-tile ping-pong
    {(0, 1): (0, 2), (0, 2): (1, 2), (1, 2): (0, 2)},  # hand 3: enters via (0,1), settles on (0,2)<->(1,2)
    {(4, 2): (4, 2)},  # hand 4: single tile, self-loop
]

# Each hand's *steady-state* tiles (excluding one-time entry keys, like
# hand 3's (0,1)) must be pairwise disjoint -- that's what makes a
# permanent merge structurally impossible once a hand settles into its zone.
STEADY_STATE = [
    {(0, 0), (1, 0), (1, 1), (0, 1)},
    {(2, 0), (3, 0), (3, 1), (2, 1)},
    {(4, 0), (4, 1)},
    {(0, 2), (1, 2)},
    {(4, 2)},
]
for i, a in enumerate(STEADY_STATE):
    for j, b in enumerate(STEADY_STATE):
        if i < j:
            assert not (a & b), f"zones {i} and {j} overlap: {a & b}"
print("all 5 zones pairwise disjoint")
print("total tiles covered:", sum(len(z) for z in STEADY_STATE), "of 13 free tiles")


def assert_is_closed_adjacent_cycle_allow_self(loop_order):
    """Same check as assert_is_closed_adjacent_cycle, but also accepts a
    single tile mapping to itself (hand 4's zone -- nowhere else to go)."""
    if len(loop_order) == 1:
        ((tile, target),) = loop_order.items()
        assert tile == target
        return
    assert_is_closed_adjacent_cycle(loop_order)


for i, zone in enumerate(HAND_ZONES):
    home_only = {k: v for k, v in zone.items() if k in STEADY_STATE[i]}
    assert_is_closed_adjacent_cycle_allow_self(home_only)
    print(f"zone {i}: valid, {len(home_only)} tile(s)")

all 5 zones pairwise disjoint
total tiles covered: 13 of 13 free tiles
zone 0: valid, 4 tile(s)
zone 1: valid, 4 tile(s)
zone 2: valid, 2 tile(s)
zone 3: valid, 2 tile(s)
zone 4: valid, 1 tile(s)


In [14]:
def wheat_loop_hands_zones_agent(obs):
    wheat_seeds = obs["private"]["seeds"].get("WHEAT", 0)

    market = []
    hands = []

    for i, (hx, hy) in enumerate(obs["farms"][obs["player"]]["hands"]):
        hand_loop = {**LOOP_ORDER_HANDS, **SPAWN_FEEDER, **HAND_ZONES[i % len(HAND_ZONES)]}
        target_x, target_y = hand_loop[(hx, hy)]
        if (target_x, target_y) == (hx, hy):
            move = ["PASS"]
        else:
            move = [DIRECTIONS[(target_x - hx, target_y - hy)]]

        hand_tile = obs["farms"][obs["player"]]["tiles"][hy][hx]
        if hand_tile == "LOCKED":
            op = move
        elif hand_tile is None and wheat_seeds > 0:
            op = ["PLANT", "WHEAT"]
        elif hand_tile is not None and hand_tile.get("kind") == "PLANT":
            crop_age = obs["day"] - hand_tile["planted_day"]
            if crop_age >= 2:
                op = ["HARVEST"]
            elif not hand_tile.get("watered_today"):
                op = ["WATER"]
            else:
                op = move
        else:
            op = move
        hands.append(op)

    if len(obs["farms"][obs["player"]]["hands"]) < 5:
        market.append(["HIRE"])

    if obs["step"] == 0 or wheat_seeds <= 5:
        market.append(["BUY_SEED", "WHEAT", 6 - wheat_seeds])

    farmer_pos = obs["farms"][obs["player"]]["farmer"]
    x, y = farmer_pos
    target_x, target_y = LOOP_ORDER_12[(x, y)]
    direction = DIRECTIONS[(target_x - x, target_y - y)]

    tile = obs["farms"][obs["player"]]["tiles"][y][x]

    wheat_in_shed = obs["private"]["shed"].get("WHEAT", 0)
    if wheat_in_shed > 0:
        market.append(["SELL", "WHEAT", 1])

    if tile is None and wheat_seeds > 0:
        farmer = ["PLANT", "WHEAT"]
    elif tile is not None and tile.get("kind") == "PLANT":
        crop_age = obs["day"] - tile["planted_day"]
        if crop_age >= 2:
            farmer = ["HARVEST"]
        elif not tile.get("watered_today"):
            farmer = ["WATER"]
        else:
            farmer = [direction]
    else:
        farmer = [direction]

    return {"farmer": farmer, "hands": hands, "market": market}

In [15]:
hands_log_zones = hand_position_log(wheat_loop_hands_zones_agent)
print(
    "max streak of ALL hands glued to the same tile (zones version):",
    max_full_collapse_streak(hands_log_zones),
)

from itertools import chain

tended_before = set(chain.from_iterable(hands_log))
tended_after = set(chain.from_iterable(hands_log_zones))
print("unique tiles ever occupied by a hand -- before (shared loop):", len(tended_before))
print("unique tiles ever occupied by a hand -- after (zones):", len(tended_after))

max streak of ALL hands glued to the same tile (zones version): 17
unique tiles ever occupied by a hand -- before (shared loop): 13
unique tiles ever occupied by a hand -- after (zones): 16


## Validate against the current best

Per this project's rule (recompute the baseline in the same run): the
agent to beat is `wheat_loop_hands_agent` (the clumping version, already
graduated to `src/`), not further back. Run twice, per the harness note
in `PROGRESS.md` (2026-08-19).

In [16]:
print(evaluate(wheat_loop_hands_agent, wheat_loop_hands_zones_agent, n=20))

{'win_rate': 0.0, 'avg_money': 6003.0}


In [17]:
print(evaluate(wheat_loop_hands_agent, wheat_loop_hands_zones_agent, n=20))

{'win_rate': 0.0, 'avg_money': 6209.0}
